In [ ]:
import os
import pandas

# Construction du csv emotion

In [ ]:
FOLDER_EMOTION = "sentence_level_all_features"
NO_FEATURES_MESSAGE = "No features found."

In [ ]:
def extract_text(lines):
    txt = ("\n").join(lines[3:-3])
    return txt.split("Sentence: ")[1].strip()

def extract_cats(lines):
    return lines[-1].split(": ")[1].replace("[", "").replace("]", "").replace(" ", "").split(",")

def extract_tokens(lines):
    return lines[-3].split(": ")[1].strip().replace("[", "").replace("]", "").replace(" ", "").split(",")

def extract_lemmas(lines):
    return lines[-2].split(": ")[1].strip().replace("[", "").replace("]", "").replace(" ", "").split(",")

In [ ]:
def extract_from_file(lines,id):

    if lines[-1] == NO_FEATURES_MESSAGE:
        print("No features found for file id: ", id)
        return {"text": ("\n").join(lines[3:-1]).split("Sentence: ")[1].strip(),
                "Tokens": [],
                "Lemmas": [],
                "Cats": [],
                "file_id": id,
                "feature_found": False}

    else:
        return {"text": extract_text(lines),
                "Tokens": extract_tokens(lines),
                "Lemmas": extract_lemmas(lines),
                "Cats": extract_cats(lines),
                "file_id": id,
                "feature_found": True}


In [ ]:
rows = []
for filename in os.listdir(FOLDER_EMOTION):
    if filename.endswith('.txt'):
        path = os.path.join(FOLDER_EMOTION, filename)
        with open(path, 'r', encoding='utf-8') as txt:
            lines = txt.readlines()
            rows.append(extract_from_file(lines, int(filename.split(".")[0])))

In [ ]:
with open(os.path.join(FOLDER_EMOTION, "6.txt"), 'r', encoding='utf-8') as f_out:
    print(f_out.read())

In [ ]:
print(len(rows), "rows extracted.")
df = pandas.DataFrame(rows)

In [ ]:
df.columns

In [ ]:
df['feature_found'].value_counts()

In [ ]:
df = df.set_index("file_id")
df = df.sort_index()
df.head()

In [ ]:
df.to_csv("corpus_emotion_haineux.csv", index=True)

In [ ]:
df.iloc[0]['text']

# Creation du csv finale

In [ ]:
FOLDER_BRUT = "corpus_brut_haineux"
df_brut = pandas.read_csv(f"{FOLDER_BRUT}/corpus_haineux.csv",sep=";")
df_brut.head(0)

In [ ]:
df.columns

In [ ]:
df_brut["Lemmas"] = None
df_brut["Tokens"] = None
df_brut["Cats"] = None
df_brut["Matching_id"] = None

In [ ]:
index = 0
for idx,row in df_brut.iterrows():

    lemmas = []
    tokens = []
    cats = []
    start_index = index
    culmative_length = 0
    lenght_of_ortignal_tweet = len(row["tweet"].strip())

    while culmative_length < lenght_of_ortignal_tweet :
        len_of_text = len(df.iloc[index]['text'].strip())

        culmative_length += len_of_text + 1

        lemmas += df.iloc[index]['Lemmas']
        tokens += df.iloc[index]['Tokens']
        cats += df.iloc[index]['Cats']

        index += 1

    if culmative_length != lenght_of_ortignal_tweet:
        raise ValueError(f"Length mismatch for row {idx}: cumulative length {culmative_length} does not match original tweet length {lenght_of_ortignal_tweet}.")

    df_brut.at[idx, "Lemmas"] = lemmas
    df_brut.at[idx, "Tokens"] = tokens
    df_brut.at[idx, "Cats"] = cats
    df_brut.at[idx, "Matching_id"] = range(start_index, index+1)




In [ ]:
pj = len(df_brut.iloc[0]['tweet'])
print(df_brut.iloc[0]['tweet'])
print("longueur du texte :", pj)
lent = 0
index = 0
while lent < pj:
    lentt = len(df.iloc[index]['text'].strip())
    print(df.iloc[index]['text'])
    lent += lentt + 1
    index += 1
print("longueur cumulée :", lent)
if lent == pj:
    print("index du texte :", index)
    df.iloc[0:index-1]['text']